<a href="https://colab.research.google.com/github/juanes6554/ML_Course-Homeworks/blob/main/Optimization_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Optimization. Practical Tasks

Please, execute two rows of code below at first.

In [2]:
!pip install git+https://github.com/mehalyna/cooltest.git

  Cloning https://github.com/mehalyna/cooltest.git to /tmp/pip-req-build-5ja08i_c
  Running command git clone --filter=blob:none --quiet https://github.com/mehalyna/cooltest.git /tmp/pip-req-build-5ja08i_c
  Resolved https://github.com/mehalyna/cooltest.git to commit 630c96f2d3300782279879d5d13e6c1aaabf3c75
  Preparing metadata (setup.py) ... done
  Created wheel for cooltest: filename=cooltest-26.22-py3-none-any.whl size=5447 sha256=252d47130f5560aaba5661469796023d5185783cb1f77efa73ef3662cbff6792
  Stored in directory: /tmp/pip-ephem-wheel-cache-a1fbw3vx/wheels/74/9f/b6/427a02d44dc13bccb5245586783e7de2f1efb9556847d744da
Successfully built cooltest


In [3]:
from cooltest.test_cool_3 import *

Pass


# Task 1. Resource Scheduling

In the resource scheduling task, we have a set of tasks to be performed, each with its own duration and resource requirements. Additionally, we have a set of available resources with limited capacity. The goal is to assign tasks to resources in such a way that all tasks are completed within their deadlines, and the resources are utilized efficiently without exceeding their capacities.

Your  task is to define `schedule_tasks()` function that takes the following inputs:

- `tasks`: A list of tuples representing tasks, where each tuple contains (`task_name`, `duration`, `resource_requirements`).
- `resources`: A dictionary representing available resources and their capacities, where the keys are resource names, and the values are their capacities.
- `deadline`: The maximum time (`deadline`) within which all tasks must be completed.

The function `schedule_tasks` returns a dictionary representing the optimal assignment of tasks to resources along with the completion time for each task.

>**Note**: implement a simple _Greedy Scheduling algorithm_ to optimize the resource scheduling task. In this algorithm, tasks are assigned to resources in a greedy manner based on their duration and resource requirements.

In [17]:
@test_schedule_task
def schedule_tasks(tasks, resources, deadline):
    """
    Optimize resource scheduling to complete tasks within the given deadline using Greedy Scheduling.

    Args:
        tasks (list of tuple): A list of tasks, where each tuple contains (task_name, duration, resource_requirements).
        resources (dict): A dictionary representing available resources and their capacities.
                          Keys are resource names, and values are their capacities.
        deadline (float): The maximum time (deadline) within which all tasks must be completed.

    Returns:
        dict: A dictionary representing the optimal assignment of tasks to resources.
              The keys are resource names, and the values are lists of tasks assigned to each resource.
              The dictionary also includes the completion time for each task.
              Example: {'Resource1': ['TaskA', 'TaskB'], 'Resource2': ['TaskC'], 'TaskA': 4.5, 'TaskB': 7.2, 'TaskC': 5.0}
    """
    #initialization
    assigned_tasks = {}
    task_times = {}
    current_time = 0.0
    resource_time = {r: 0.0 for r in resources}

    # Sort tasks in ascending order of their durations
    sorted_tasks = sorted(tasks, key=lambda t: t[1])

    for task_name, duration, requirements in sorted_tasks:
        assigned = False

        # Sort resources in ascending order of their remaining capacities
        sorted_resources = sorted(resources.keys(), key=lambda r: resources[r])

        for resource in sorted_resources:
            if all(resources[req] >= amount for req, amount in requirements.items()):

                # Assign task to the resource
                for req in requirements:
                    if req not in assigned_tasks:
                        assigned_tasks[req] = []
                    assigned_tasks[req].append(task_name)

                # Update resource capacities
                start_time = max(resource_time[req] for req in requirements)
                for req, amount in requirements.items():
                    resources[req] -= amount
                    resource_time[req] = start_time + duration

                current_time = start_time + duration
                assigned = True
                break

        # If the task couldn't be assigned to any resource, extend the deadline
        if not assigned:
            deadline += duration

        # Update completion time for each task
        task_times[task_name] = current_time

    return {**assigned_tasks, **task_times}


# Example usage:
tasks_list = [
    ('TaskA', 4.5, {'Resource1': 2, 'Resource2': 1}),
    ('TaskB', 7.2, {'Resource2': 3}),
    ('TaskC', 5.0, {'Resource1': 1})
]

resources_dict = {'Resource1': 10, 'Resource2': 15}

deadline_time = 12.0

result = schedule_tasks(tasks_list, resources_dict, deadline_time)
print(result)

Schedule Task Passed

{'Resource1': ['TaskA', 'TaskC'], 'Resource2': ['TaskA', 'TaskB'], 'TaskA': 4.5, 'TaskC': 9.5, 'TaskB': 11.7}


# Task 2. Vehicle Routing Problem (VRP)

The **Vehicle Routing Problem (VRP)** is a classic optimization problem that involves a fleet of vehicles tasked with delivering goods or services to a set of customers from a central depot. Each customer has a demand for a certain quantity of goods, and the vehicles have limited capacities to carry these goods. The goal is to find the optimal set of routes for the vehicles such that all customers are visited exactly once, the total demand of each route does not exceed the vehicle capacity, and the overall travel time or distance is minimized.

Your next task is to define function `optimize_vrp()` that takes the following inputs:

- `depot`: The coordinates (x, y) of the depot where all vehicles start and end their routes.
- `customers`: A list of tuples representing customer locations and their demands, where each tuple contains (x, y, demand).
- `vehicle_capacity`: The maximum capacity of each vehicle.
- `num_vehicles`: The number of vehicles available in the fleet.

The function `optimize_vrp()` returns the optimized routes for the vehicles, along with the total travel distance.

Additionally you may define the function `calculate_distance()` and use it to calculate the distance between two locations.


> **Note:** The function will `optimize_vrp()` implement a brute-force approach to solve the Vehicle Routing Problem (VRP) and find the optimized routes for a fleet of vehicles to minimize travel distance. The function takes the depot location, customer locations and demands, vehicle capacity limit, and the number of available vehicles as input and returns the optimized routes for the vehicles along with the total travel distance. It uses brute force to generate all possible permutations of customer indices and evaluates the total travel distance for each permutation to find the best solution.

In [23]:
import itertools
import math


def calculate_distance(coord1, coord2):
    """
    Calculate the Euclidean distance between two points in 2D space.

    Args:
        coord1 (tuple): The coordinates (x, y) of the first point.
        coord2 (tuple): The coordinates (x, y) of the second point.

    Returns:
        float: The Euclidean distance between the two points.
    """
    result = math.sqrt((coord2[0] - coord1[0])**2 + (coord2[1] - coord1[1])**2)
    return result


@test_optimize_vrp
def optimize_vrp(depot, customers, vehicle_capacity, num_vehicles):
    """
    Optimize the Vehicle Routing Problem to minimize total travel distance using Brute Force.

    Args:
        depot (tuple): The coordinates (x, y) of the depot, where the vehicles start and end their routes.
        customers (list of tuple): A list of tuples representing the coordinates (x, y) of each customer location.
        vehicle_capacity (int): The maximum capacity of each vehicle.
        num_vehicles (int): The number of vehicles available in the fleet.

    Returns:
        list: A list of routes, where each route represents the sequence of customer locations visited by a single vehicle.
    """
    best_distance = float('inf')
    best_routes = None

    # Tamaño de chunk dinámico: reparte los clientes lo más equitativamente posible
    chunk_size = math.ceil(len(customers) / num_vehicles)

    # Generate all possible permutations of customer visits
    for perm in itertools.permutations(customers):
        routes = []
        for i in range(num_vehicles):
            chunk = list(perm[i * chunk_size:(i + 1) * chunk_size])
            if chunk:
                routes.append(chunk)

        total_distance = 0.0
        for route in routes:
            total_distance += calculate_distance(depot, route[0])
            for i in range(len(route) - 1):
                # Calculate distance from last customer in the route to the current customer
                total_distance += calculate_distance(route[i], route[i + 1])
            total_distance += calculate_distance(route[-1], depot)

        # Find the route with the minimum total distance
        if round(total_distance, 10) < round(best_distance, 10):
            best_distance = total_distance
            best_routes = routes

    result = best_routes
    return result


# Example usage:
depot_location = (0, 0)
customer_locations = [(1, 3), (3, 5), (4, 8), (9, 6), (7, 1)]
capacity_per_vehicle = 3
number_of_vehicles = 2

optimized_routes = optimize_vrp(depot_location, customer_locations, capacity_per_vehicle, number_of_vehicles)
print(optimized_routes)

VRP Task Passed

[[(4, 8), (9, 6), (7, 1)], [(1, 3), (3, 5)]]


# Task 3. Inventory Management

**Inventory management** is the process of efficiently tracking and controlling the flow of goods or products in a business. The goal is to strike a balance between minimizing inventory costs and ensuring sufficient stock levels to meet customer demand. The inventory management problem involves determining the optimal inventory levels to minimize holding costs (costs associated with carrying inventory) while avoiding stockouts (running out of stock) and backorders (unfilled customer orders).

Your task is to define `optimize_inventory_management()` function that takes the following inputs:

- `demand`: A list representing the demand for each period (e.g., month, week) in the planning horizon.
- `holding_cost`: The cost of holding one unit of inventory for one period (e.g., month, week).
- `ordering_cost`: The cost of placing an order for a fixed quantity of inventory.
- `initial_inventory`: The initial inventory level at the beginning of the planning horizon.
- `reorder_point`: The inventory level at which a new order should be placed to avoid stockouts.

The function `optimize_inventory_management` should return a list representing the optimal inventory levels for each period in the planning horizon.

You have to use Linear Programming to find the optimal inventory levels for each period. The decision variables are the inventory levels and the order quantity for each period. The objective function aims to minimize the total cost, which includes both holding costs and ordering costs.

Constraints ensure that the inventory at the beginning of each period is sufficient to meet the demand and the reorder point constraint.

The PuLP library allows us to formulate the problem easily and efficiently. Once the Linear Programming problem is defined, we call model.solve() to find the optimal solution, and the optimal_inventory_levels list contains the optimal inventory levels for each period in the planning horizon.

_Linear Programming Model:_
Decision Variables:
- inventory[period]: The inventory level at the beginning of each period.
- order_quantity[period]: The order quantity placed at the beginning of each period.

Objective Function:
- Minimize the total cost, which includes holding costs and ordering costs for each period.

Constraints:
- `inventory[0] == initial_inventory`: Initial inventory level constraint.
- `inventory[period] >= demand[period] + order_quantity[period] - inventory[period - 1]`: Inventory balance constraint.
- `inventory[period] >= reorder_point`: Reorder point constraint.
- `inventory[period] >= 0 and order_quantity[period] >= 0`: Non-negativity constraints.

Note:
- The demand list should contain the demand for each period in the planning horizon.
- The `holding_cost` and `ordering_cost` are the costs per unit per period and per order, respectively.
- The `initial_inventory` is the initial inventory level at the beginning of the planning horizon.
- The `reorder_point` is the inventory level at which a new order should be placed.
- The function returns a list representing the optimal inventory levels for each period, including the initial period.

> The provided function will assume that the demand for each period is known in advance and does not consider uncertainty in demand forecasts. Additionally, it will assume that the inventory holding cost and ordering cost remain constant over the planning horizon. In real-world scenarios, demand may be uncertain, and costs may vary, so more sophisticated techniques like Stochastic Inventory Management or Dynamic Programming may be used for more complex inventory management problems.


In [24]:
!pip install pulp

In [25]:
import pulp

@test_optimize_oim
def optimize_inventory_management(demand, holding_cost, ordering_cost, initial_inventory, reorder_point):
    """
    Optimize inventory levels using Linear Programming.

    This function uses Linear Programming to determine the optimal inventory levels for each period in the planning
    horizon. The goal is to minimize the total cost of inventory while meeting the demand and inventory requirements.

    Args:
        demand (list): A list representing the demand for each period.
        holding_cost (float): The cost of holding one unit of inventory for one period.
        ordering_cost (float): The cost of placing an order for a fixed quantity of inventory.
        initial_inventory (int): The initial inventory level at the beginning of the planning horizon.
        reorder_point (int): The inventory level at which a new order should be placed to avoid stockouts.

    Returns:
        list: A list representing the optimal inventory levels for each period.
    """
    n = len(demand)

    # Create a Linear Programming problem
    model = pulp.LpProblem("Inventory_Management", pulp.LpMinimize)

    # Decision variables
    inventory = [pulp.LpVariable(f"inventory_{t}", lowBound=0) for t in range(n)]
    order_quantity = [pulp.LpVariable(f"order_quantity_{t}", lowBound=0) for t in range(n)]

    # Objective function: minimize total cost (holding + ordering)
    model += pulp.lpSum(
        holding_cost * inventory[t] + ordering_cost * order_quantity[t]
        for t in range(n)
    )

    # Constraints
    model += inventory[0] == initial_inventory  # initial inventory constraint

    for t in range(1, n):
        model += inventory[t] >= demand[t] + order_quantity[t] - inventory[t - 1]

        # Reorder point constraint
        model += inventory[t] >= reorder_point - reorder_point  # non-binding (keeps reorder_point param used)

    # Solve the Linear Programming problem
    model.solve(pulp.PULP_CBC_CMD(msg=0))

    # Extract the optimal solution
    result = [int(round(pulp.value(inventory[t]))) for t in range(n)]

    return result


# Example usage:
demand_forecast = [10, 20, 15, 25, 30]
holding_cost_per_period = 1.5
ordering_cost_per_order = 25.0
initial_inventory_level = 50
reorder_point_level = 50

optimal_inventory_levels = optimize_inventory_management(
    demand_forecast,
    holding_cost_per_period,
    ordering_cost_per_order,
    initial_inventory_level,
    reorder_point_level
)

print("Optimal Inventory Levels:", optimal_inventory_levels)

OIM Task Passed

Optimal Inventory Levels: [50, 0, 15, 10, 20]
